# Phase 2 — Baseline CNN Classifier

Trains `BaselineCNN` (src/models/model.py) on the NEU-DET dataset using the
reusable training loop in `src/models/train.py`. Same logic as
`src/models/run_training.py` (the CLI script), but here we can inspect
curves and the confusion matrix interactively.

In [ ]:
import sys
sys.path.append('..')

import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

from src.data.dataset import NEUDataset, CLASS_NAMES
from src.models.model import BaselineCNN
from src.models.train import fit, evaluate

DATA_ROOT = '../data/raw/NEU-DET'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
train_set = NEUDataset(f'{DATA_ROOT}/train')
val_set = NEUDataset(f'{DATA_ROOT}/validation')
print(f'Train samples: {len(train_set)} | Val samples: {len(val_set)}')

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)

In [ ]:
model = BaselineCNN(num_classes=len(CLASS_NAMES)).to(device)
print(model)

In [ ]:
history = fit(
    model,
    train_loader,
    val_loader,
    device,
    epochs=15,
    lr=1e-3,
    checkpoint_path='../models/baseline_cnn.pt',
)

## Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['val_acc'], label='val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## Confusion matrix on the validation set

In [ ]:
import torch.nn as nn
from sklearn.metrics import confusion_matrix, classification_report

# Load best checkpoint before final evaluation
model.load_state_dict(torch.load('../models/baseline_cnn.pt', map_location=device))

val_loss, val_acc, y_true, y_pred = evaluate(model, val_loader, nn.CrossEntropyLoss(), device)
print(f'Best checkpoint — val_loss={val_loss:.4f} val_acc={val_acc:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax.text(j, i, cm[i, j], ha='center', va='center')
plt.colorbar(im)
plt.tight_layout()
plt.show()